# Chatbot - Gradio X llama-cpp-python

In [1]:
import gradio as gr
from IPython.display import IFrame, display
import os
from llama_cpp import Llama
import json

In [2]:
gr.close_all()


## Running a Gradio App on JupyterHub

This code demonstrates how to run a **Gradio app** (a simple web-based interface for Python functions) inside a **JupyterHub environment** such as DataHub or CloudBank.

Gradio normally launches on `localhost`, but on JupyterHub the server runs behind a proxy — so we use the environment variable `JUPYTERHUB_SERVICE_PREFIX` to route the app correctly through the proxy system.



In [3]:
demo1 = gr.Interface(fn=lambda x: f"Hello {x}!", inputs="text", outputs="text")

base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
demo1.launch(
    share=False,
    prevent_thread_lock=True,
    server_port=7860,
    root_path=f"{base_url}proxy/7860",
    inline=False
)

proxy_url = f"{base_url}proxy/7860/"
display(IFrame(src=proxy_url, width=1000, height=600))

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/srv/conda/envs/notebook/lib/python3.12/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [4]:
demo1.close()

Closing server running on port: 7860


In [5]:
print(base_url)


/user/rscherl@monmouth.edu/


## Set up the llama-cpp-python framework

and Run llama-cpp-python behind the Chatbot

In [6]:
path= "/home/jovyan/shared/models/gguf/"

In [8]:
model = Llama(
    model_path=os.path.join(path, "qwen2.5-0.5b-instruct-q8_0.gguf"),
    n_ctx=2048,
    n_threads=None,
    verbose=True,
    chat_format="chatml"
)

llama_model_loader: loaded meta data with 26 key-value pairs and 291 tensors from /home/jovyan/shared/models/gguf/qwen2.5-0.5b-instruct-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-0.5b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-0.5b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 630M
llama_model_loader: - kv   6:                          qwen2.block_count u32              = 24
llama_model_load

In [9]:
# Define function to call the model
def chat_with_model(prompt):
    messages = [{"role": "user", "content": prompt}]
    response = model.create_chat_completion(messages=messages, max_tokens=256)
    return response["choices"][0]["message"]["content"]

# Define Gradio interface
demo2 = gr.Interface(fn=chat_with_model, inputs="text", outputs="text", title="Small Model Chat")

# Launch on JupyterHub proxy
base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
demo2.launch(
    share=False,
    prevent_thread_lock=True,
    server_port=7860,
    root_path=f"{base_url}proxy/7860",
    inline=False
)

# Display inline in the notebook
proxy_url = f"{base_url}proxy/7860/"
display(IFrame(src=proxy_url, width=1000, height=600))

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/srv/conda/envs/notebook/lib/python3.12/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
llama_perf_context_print:        load time =     256.97 ms
llama_perf_context_print: prompt eval time =     256.69 ms /    19 tokens (   13.51 ms per token,    74.02 tokens per second)
llama_perf_context_print:        eval time =    1739.45 ms /    40 runs   (   43.49 ms per token,    23.00 tokens per second)
llama_perf_context_print:       total time =    2061.82 ms /    59 tokens
llama_perf_context_print:    graphs reused =         39
/srv/conda/envs/notebook/lib/python3.12/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Llama.generate: 8 prefix-match hit, remaining

In [10]:
demo2.close()

Closing server running on port: 7860


## Lets Build the History of the Chat 

This will be a Json file that stores the chat history 

In [11]:
demo2.close()

In [12]:
# ===== Persistent History File =====
HISTORY_FILE = "chat_history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r") as f:
                content = f.read().strip()
                if not content:
                    return []  # empty file
                return json.loads(content)
        except json.JSONDecodeError:
            print("⚠️ Warning: history file is corrupted or empty, resetting it.")
            return []
    return []

def save_history(history):
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=2)

history = load_history()



In [13]:
# ===== Chat Function =====
def chat_with_model(user_input):
    """Append user input to history, generate response, and persist conversation."""
    # Reload history each time in case file changed externally
    global history
    history = load_history()

    # Build messages list from history
    messages = []
    for h in history:
        messages.append({"role": "user", "content": h['user']})
        messages.append({"role": "assistant", "content": h['model']})
    messages.append({"role": "user", "content": user_input})

    # Generate model response
    response = model.create_chat_completion(messages=messages, max_tokens=100)
    response_text = response["choices"][0]["message"]["content"]

    # Update and persist history
    history.append({"user": user_input, "model": response_text})
    save_history(history)

    return response_text


In [14]:
# ===== Gradio Interface =====
demo3 = gr.Interface(
    fn=chat_with_model,
    inputs="text",
    outputs="text",
    title="Persistent Small Model Chat",
    description="Chat with a local llama-cpp-python model that remembers previous conversations."
)


base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
demo3.launch(
    share=False,
    prevent_thread_lock=True,
    server_port=7860,
    root_path=f"{base_url}proxy/7860",
    inline=False
)

proxy_url = f"{base_url}proxy/7860/"
display(IFrame(src=proxy_url, width=1000, height=600))

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/srv/conda/envs/notebook/lib/python3.12/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Llama.generate: 11 prefix-match hit, remaining 8 prompt tokens to eval
llama_perf_context_print:        load time =     256.97 ms
llama_perf_context_print: prompt eval time =     167.24 ms /     8 tokens (   20.90 ms per token,    47.84 tokens per second)
llama_perf_context_print:        eval time =     517.47 ms /    10 runs   (   51.75 ms per token,    19.32 tokens per second)
llama_perf_context_print:       total time =     693.81 ms /    18 tokens
llama_perf_context_print:    graphs reused =          9


## Now we can check the history that we have 

In [15]:
for turn in json.load(open("chat_history.json")):
    print(f"🧑 User: {turn['user']}\n🤖 Model: {turn['model']}\n")

🧑 User: How old is Melania Trump?
🤖 Model: Melania Trump is 44 years old.



In [16]:
#if you want to clear the history 
#open("chat_history.json", "w").write("[]")

In [17]:
demo3.close()

Closing server running on port: 7860


In [18]:
gr.close_all()
